# Задание 5 — Реализация и анализ на базов модел

**Цел**: Създаване на базови модели (Base Models) и техния анализ, който да послужи като отправна точка (benchmark) за по-сложни модели (Невронни мрежи).

**CRISP-DM фаза**: Modeling -> Evaluation -> Data Understanding

**Критерии**:
1. Установяване на наивен benchmark (Majority Class).
2. Подбор на поне два базови модела (Logistic Regression, Decision Tree).
3. Коректна имплементация (без изтичане на информация).
4. Коректно валидиране (Train/Val/Test split).
5. Избор и представяне на метрики (Accuracy, Precision, Recall, F1, ROC-AUC).
6. Документиране на хиперпараметри и резултати.
7. Сравнителна таблица.
8. Анализ на грешките.
9. Интерпретация на резултатите.
10. Избор на най-добър базов модел.

## 1. Setup и Зареждане на Данните

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import PredefinedSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix, roc_curve
)
from sklearn.base import BaseEstimator, TransformerMixin

# Настройки за визуализация
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Пътища към данните
DATA_DIR = '../data/processed_v9'

def load_data(data_dir):
    try:
        X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
        y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
        X_val = pd.read_csv(os.path.join(data_dir, 'X_val.csv'))
        y_val = pd.read_csv(os.path.join(data_dir, 'y_val.csv'))
        X_test = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
        y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))
        
        # Уверяваме се, че y са вектори
        y_train = y_train.values.ravel()
        y_val = y_val.values.ravel()
        y_test = y_test.values.ravel()
        
        return X_train, y_train, X_val, y_val, X_test, y_test
    except FileNotFoundError as e:
        print(f"Error loading data: {e}")
        return None, None, None, None, None, None

X_train, y_train, X_val, y_val, X_test, y_test = load_data(DATA_DIR)

if X_train is not None:
    print(f"Train set: {X_train.shape}")
    print(f"Val set: {X_val.shape}")
    print(f"Test set: {X_test.shape}")

### Preprocessing Pipeline
Дефинираме pipeline за обработка на данните, включващ импутация, скалиране и One-Hot Encoding.

In [ ]:
# Дефиниране на характеристиките
numeric_features = ['Age', 'Years_At_Company', 'Monthly_Salary', 'Work_Hours_Per_Week', 
                    'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 
                    'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions']
categorical_features = ['Department', 'Gender', 'Job_Title', 'Education_Level']

# Pipeline за числови данни
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline за категорийни данни
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Общ Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit на preprocessor върху Train сет-а
X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

# Запазваме имената на колоните за по-късна интерпретация
ohe_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
feature_names = numeric_features + list(ohe_feature_names) 

print(f"Processed Train shape: {X_train_prep.shape}")

## 2. Установяване на Наивен Benchmark (Criterion 1)
Използваме `DummyClassifier` със стратегия `most_frequent` (винаги предсказва мнозинствения клас), за да установим базовото ниво на точност, което всеки модел трябва да надмине.

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train_prep, y_train)

y_val_pred_dummy = dummy_clf.predict(X_val_prep)
y_val_proba_dummy = dummy_clf.predict_proba(X_val_prep)[:, 1]

# Метрики
def calculate_metrics(y_true, y_pred, y_proba, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else 0.5
    }

dummy_metrics = calculate_metrics(y_val, y_val_pred_dummy, y_val_proba_dummy, "Naive Benchmark")
print(dummy_metrics)

## 3. Базови Модели (Criteria 2, 4, 6)

### 3.1 Logistic Regression
Логистичната регресия е отличен първи модел заради своята простота и възможност за интерпретация на коефициентите.
Ще тестваме различни стойности на регуляризация (`C`).

In [ ]:
results_lr = []
C_values = [0.01, 0.1, 1, 10]

best_lr_model = None
best_lr_f1 = -1

for c in C_values:
    lr = LogisticRegression(C=c, max_iter=1000, random_state=42)
    lr.fit(X_train_prep, y_train)
    
    y_pred = lr.predict(X_val_prep)
    y_proba = lr.predict_proba(X_val_prep)[:, 1]
    
    metrics = calculate_metrics(y_val, y_pred, y_proba, f"Logistic Regression (C={c})")
    results_lr.append(metrics)
    
    if metrics['F1 Score'] > best_lr_f1:
        best_lr_f1 = metrics['F1 Score']
        best_lr_model = lr

pd.DataFrame(results_lr)

### 3.2 Decision Tree
Дървото на решенията е нелинеен модел, който също е силно интерпретируем. 
Ще варираме `max_depth`, за да контролираме overfitting-а.

In [ ]:
results_dt = []
depths = [3, 5, 10, None]

best_dt_model = None
best_dt_f1 = -1

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_prep, y_train)
    
    y_pred = dt.predict(X_val_prep)
    y_proba = dt.predict_proba(X_val_prep)[:, 1]
    
    model_name = f"Decision Tree (Depth={depth})"
    metrics = calculate_metrics(y_val, y_pred, y_proba, model_name)
    results_dt.append(metrics)
    
    if metrics['F1 Score'] > best_dt_f1:
        best_dt_f1 = metrics['F1 Score']
        best_dt_model = dt

pd.DataFrame(results_dt)

## 4. Сравнителна Таблица (Criterion 7)
Сравняваме най-добрите конфигурации на моделите с бейзлайна.

In [ ]:
# Събираме най-добрите резултати
final_comparison = []
final_comparison.append(dummy_metrics)

# Най-добър LR
y_val_pred_lr = best_lr_model.predict(X_val_prep)
y_val_proba_lr = best_lr_model.predict_proba(X_val_prep)[:, 1]
final_comparison.append(calculate_metrics(y_val, y_val_pred_lr, y_val_proba_lr, "Best Logistic Regression"))

# Най-добър DT
y_val_pred_dt = best_dt_model.predict(X_val_prep)
y_val_proba_dt = best_dt_model.predict_proba(X_val_prep)[:, 1]
final_comparison.append(calculate_metrics(y_val, y_val_pred_dt, y_val_proba_dt, "Best Decision Tree"))

comparison_df = pd.DataFrame(final_comparison).set_index('Model')
display(comparison_df)

# Визуализация
comparison_df[['Accuracy', 'F1 Score', 'ROC-AUC']].plot(kind='bar', figsize=(10, 6))
plt.title("Сравнение на метриките на моделите")
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

## 5. Анализ на Грешките (Criterion 8)
Разглеждаме матриците на объркването (Confusion Matrices), за да видим какви грешки правят моделите (False Positives vs False Negatives).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LR Confusion Matrix
cm_lr = confusion_matrix(y_val, y_val_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title("Logistic Regression Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# DT Confusion Matrix
cm_dt = confusion_matrix(y_val, y_val_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title("Decision Tree Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.show()

## 6. Интерпретация на Резултатите (Criterion 9)

In [ ]:
# 6.1 Logistic Regression Coefficients
coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': best_lr_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Coefficient', y='Feature', data=coefs.head(10))
plt.title("Top 10 Positive Features (Logistic Regression)")
plt.show()

plt.figure(figsize=(10, 8))
sns.barplot(x='Coefficient', y='Feature', data=coefs.tail(10))
plt.title("Top 10 Negative Features (Logistic Regression)")
plt.show()

In [ ]:
# 6.2 Decision Tree Feature Importance
importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': best_dt_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=importances.head(15))
plt.title("Top 15 Feature Importances (Decision Tree)")
plt.show()

## 7. Финален Избор (Criterion 10)
На базата на метриките (особено F1 и ROC-AUC) и интерпретируемостта, избираме най-добрия модел.

***Заключение:***
*(Тук ще се генерира автоматично заключение при изпълнение, но като цяло Дървото на решенията често улавя по-добре нелинейните зависимости, докато Логистичната регресия е по-стабилна.)*

Избраният модел е: **[Попълнете след изпълнение]**